# Market data: curves and `MarketContext`

**Prerequisites:** complete `01_foundations/core_types_and_money.ipynb` and `01_foundations/dates_calendars_schedules.ipynb` in this curriculum (finstack-quant foundations: environment, core types, and dates/money basics). This notebook assumes you can import `finstack_quant` and work with `datetime.date` and `Currency`.

## Concepts

Pricing and risk need a **consistent snapshot of the market** on a valuation date. In finstack, scalar term structures—discount factors, forward rates, credit hazards—are represented as **curves** keyed by time (year fractions from a **base date**).

The **`MarketContext`** container is the single place you **register** those curves (and FX) so downstream pricers can **`get_discount`**, **`get_forward`**, **`get_hazard`**, and read **`mc.fx`** without threading dozens of objects through every call.

- **DiscountCurve** — risk-free (or funding) discount factors \(DF(t)\); related **zero** rates via \(DF(t) = e^{-z(t)\,t}\).
- **ForwardCurve** — projection factors for one floating-rate index with an **index tenor** (e.g. 3M SOFR); `rate_between` turns them into projected fixings.
- **HazardCurve** — credit: piecewise hazard rates → survival probabilities for default timing.
- **FxMatrix** — stored FX quotes; **`FxConversionPolicy`** selects how a rate is applied for a given valuation date.

Below we walk through each API briefly, then assemble everything for a **USD corporate bond** style setup (OIS discount, SOFR forward, issuer hazard, cross-currency FX).

### DiscountCurve

Build from an identifier, **base date**, list of **(time in years, discount factor)** knots, and explicit **`interp`**, **`extrapolation`** and **`day_count`** keywords. Use **`df(t)`** and **`zero_rate(t, compounding)`** on the same time grid the curve uses internally (year fractions).

In [ ]:
from datetime import date

from finstack_quant.core.market_data import DiscountCurve

curve = DiscountCurve(
    "USD-OIS",
    date(2024, 1, 1),
    [(0.0, 1.0), (0.5, 0.975), (1.0, 0.95), (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    interp="monotone_convex",
    extrapolation="extend",
    day_count="act_365f",
)
print("id:", curve.id)
print("base_date:", curve.base_date)
print("df(1.0):", curve.df(1.0))
print("zero_rate(1.0, continuous):", curve.zero_rate(1.0, "continuous"))

### ForwardCurve

Forward curves store **projection factors** `(t, P(t))` with `P(0) = 1`, a **`base_date`**, explicit **`interp`**, **`extrapolation`** and **`day_count`**, and the index's **`index_tenor`** (e.g. `"3M"`). **`rate_between(t1, t2, accrual)`** returns the simple projected fixing over `[t1, t2]` on the index's accrual fraction; on an `act_360` curve a 0.25-year step is a 90-day period with accrual `0.25`.

In [ ]:
import math
from datetime import date

from finstack_quant.core.market_data import ForwardCurve

# Continuous projection forwards per segment: 4.25% to 1y, 4.65% to 2y, 4.90% to 5y.
fwd = ForwardCurve(
    "USD-SOFR",
    date(2024, 1, 1),
    [
        (1.0, math.exp(-0.0425)),
        (2.0, math.exp(-0.0425 - 0.0465)),
        (5.0, math.exp(-0.0425 - 0.0465 - 0.049 * 3)),
    ],
    interp="log_linear",
    extrapolation="extend",
    day_count="act_360",
    index_tenor="3M",
)
print("id:", fwd.id, "| index tenor:", fwd.index_tenor)
print("3M fixing from 1.0y:", fwd.rate_between(1.0, 1.25, 0.25))
print("3M fixing from 3.0y (interpolated):", fwd.rate_between(3.0, 3.25, 0.25))

### HazardCurve (credit)

**`HazardCurve`** models issuer default intensity over time. Knots are **(time, hazard rate)** with strictly positive times, each rate covering the segment ending at its knot; **`recovery_rate`** and **`day_count`** are required keywords that match the Rust builder.

In [ ]:
from datetime import date

from finstack_quant.core.market_data import HazardCurve

hz = HazardCurve(
    "ACME-HZD",
    date(2024, 1, 1),
    [(1.0, 0.01), (5.0, 0.03), (10.0, 0.05)],
    recovery_rate=0.40,
    day_count="act_365f",
)
print("HazardCurve id:", hz.id)
print("survival(5.0):", hz.sp(5.0))
print("hazard_rate(5.0):", hz.hazard_rate(5.0))


### FxMatrix

Store quotes with **`set_quotes(base, quote, rate)`** meaning **1 base = rate quote**. Retrieve with **`rate(base, quote, as_of_date, policy)`**; the result exposes **`.rate`** and **`.triangulated`**.

You do **not** need a direct quote for every pair. When a cross such as GBP/EUR is missing, the matrix triangulates it through its **pivot currency** (USD by default) and sets **`.triangulated = True`**. Ask the matrix for the cross rather than dividing two quotes in Python — the deep-dive notebook `market_data/fx_matrix.ipynb` covers this in full.

In [ ]:
from datetime import date

from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import FxConversionPolicy, FxMatrix

# Crosses route through an explicit USD pivot (no implicit hub).
fx = FxMatrix("USD")
fx.set_quotes([(Currency("EUR"), Currency("USD"), 1.10)])
fx.set_quotes([(Currency("GBP"), Currency("USD"), 1.27)])
fx.set_quotes([(Currency("JPY"), Currency("USD"), 0.0067)])

as_of = date(2024, 1, 1)
policy = FxConversionPolicy.CASHFLOW_DATE

eur_usd = fx.rate(Currency("EUR"), Currency("USD"), as_of, policy)
print("EUR/USD:", eur_usd.rate, "triangulated:", eur_usd.triangulated)
jpy_usd = fx.rate(Currency("JPY"), Currency("USD"), as_of, policy)
print("JPY/USD:", jpy_usd.rate, "triangulated:", jpy_usd.triangulated)

# GBP/EUR is not quoted directly: the matrix routes it through the USD pivot.
gbp_eur = fx.rate(Currency("GBP"), Currency("EUR"), as_of, policy)
print("GBP/EUR (no direct quote):", gbp_eur.rate, "triangulated:", gbp_eur.triangulated)

# A dealer-observed direct quote takes precedence over the triangulated route.
fx.set_quotes([(Currency("GBP"), Currency("EUR"), 1.1550)])
gbp_eur_direct = fx.rate(Currency("GBP"), Currency("EUR"), as_of, policy)
print(
    "GBP/EUR after direct quote:",
    gbp_eur_direct.rate,
    "triangulated:",
    gbp_eur_direct.triangulated,
)


### MarketContext

**`insert(curve)`** registers discount, forward, or hazard curves (returns **`self`** for chaining). **`insert_fx(fx)`** attaches the matrix. Retrieve curves by id with **`get_discount`**, **`get_forward`**, **`get_hazard`**.

In [ ]:
import math
from datetime import date

from finstack_quant.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    MarketContext,
)

d = DiscountCurve(
    "USD-OIS",
    date(2024, 1, 1),
    [(0.0, 1.0), (1.0, 0.95), (5.0, 0.75)],
    interp="monotone_convex",
    extrapolation="extend",
    day_count="act_365f",
)
f = ForwardCurve(
    "USD-SOFR",
    date(2024, 1, 1),
    [(1.0, math.exp(-0.0425))],
    interp="log_linear",
    extrapolation="extend",
    day_count="act_360",
    index_tenor="3M",
)

mc = MarketContext()
mc.insert(d).insert(f)
print("discount id from context:", mc.get_discount("USD-OIS").id)
print("forward id from context:", mc.get_forward("USD-SOFR").id)
print("df(1) from stored curve:", mc.get_discount("USD-OIS").df(1.0))

## Mini-example: `MarketContext` for a USD corporate bond

A typical USD fixed–float or fixed coupon bond stack needs:

- **USD-OIS** (or similar) for **risk-free / valuation discounting**
- **USD-SOFR** (3M tenor) for **floating coupon projections**
- **Issuer hazard** curve when you price **credit** (skipped if unavailable)
- **FX** if you also report notionals or covenants in EUR/GBP versus USD

We build one context, attach FX, and print a short diagnostic panel.

In [ ]:
import math
from datetime import date

from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    FxConversionPolicy,
    FxMatrix,
    MarketContext,
)

as_of = date(2024, 1, 1)

ois = DiscountCurve(
    "USD-OIS",
    as_of,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    interp="monotone_convex",
    extrapolation="extend",
    day_count="act_365f",
)
# Continuous projection forwards per segment: 5.00% to 1y, 4.65% to 3y, 4.40% to 5y.
sofr = ForwardCurve(
    "USD-SOFR",
    as_of,
    [
        (1.0, math.exp(-0.05)),
        (3.0, math.exp(-0.05 - 0.0465 * 2)),
        (5.0, math.exp(-0.05 - 0.0465 * 2 - 0.044 * 2)),
    ],
    interp="log_linear",
    extrapolation="extend",
    day_count="act_360",
    index_tenor="3M",
)

mc = MarketContext()
mc.insert(ois).insert(sofr)

from finstack_quant.core.market_data import HazardCurve

hz = HazardCurve(
    "ACME-USD-HZD",
    as_of,
    [(1.0, 0.012), (3.0, 0.018), (5.0, 0.022), (10.0, 0.028)],
    recovery_rate=0.40,
    day_count="act_365f",
)
mc.insert(hz)
hazard_line = f"survival(5y)={mc.get_hazard('ACME-USD-HZD').sp(5.0):.6f}"

fx = FxMatrix(None)
fx.set_quotes([(Currency("EUR"), Currency("USD"), 1.10)])
fx.set_quotes([(Currency("GBP"), Currency("USD"), 1.27)])
mc.insert_fx(fx)

print("--- USD rates ---")
print("OIS df(3y):", mc.get_discount("USD-OIS").df(3.0))
print("OIS zero(3y):", mc.get_discount("USD-OIS").zero_rate(3.0, "continuous"))
print("SOFR 3M fixing from 2y:", mc.get_forward("USD-SOFR").rate_between(2.0, 2.25, 0.25))
print("--- Credit ---")
print(hazard_line)
print("--- FX (vs USD) ---")
assert mc.fx is not None
print(
    "EUR/USD:",
    mc.fx.rate(
        Currency("EUR"), Currency("USD"), as_of, FxConversionPolicy.CASHFLOW_DATE
    ).rate,
)
print(
    "GBP/USD:",
    mc.fx.rate(
        Currency("GBP"), Currency("USD"), as_of, FxConversionPolicy.CASHFLOW_DATE
    ).rate,
)

# Additional MarketContext accessors (foundations)
from finstack_quant.core.market_data import PriceCurve
mc.insert_price("ACME_LAST", 55.25, "USD")
pc = PriceCurve("ACME-FWD", as_of, [(0.0, 55.0), (1.0, 56.5)], day_count="act_365f")
mc.insert(pc)
print("get_price_curve(ACME-FWD) @0.5y:", mc.get_price_curve("ACME-FWD").price(0.5))

vix = PriceCurve("VIX", as_of, [(0.0, 18.0), (1.0, 21.0)], kind="vol_index")
mc.insert(vix)
print("get_vol_index_curve(VIX) front:", mc.get_vol_index_curve("VIX").price(0.0))


## Takeaways

- **Curves** are built from **knots** in **year fractions** from a **base date**; pick **`day_count`** to match the convention of the data source.
- **`DiscountCurve.df`** and **`.zero_rate`** are the main discounting hooks; **`ForwardCurve.rate_between`** turns projection factors into fixings over a period of the curve’s **`index_tenor`**.
- **`MarketContext`** centralizes **`insert`** for curves and **`insert_fx`** for **`FxMatrix`**; pricers retrieve by **string id** via **`get_*`**.
- **FX** crosses come from the matrix: direct quote, reciprocal, or **single-hop triangulation** through the pivot currency, with **`.triangulated`** recording which. Do **not** assume multi-hop routing, and never hand-divide two quotes.
- For **corporate credit**, add a **`HazardCurve`** when the binding is available, then use **`get_hazard`** alongside OIS and SOFR for a complete USD stack.

Next in the curriculum: use this context as input to cashflow generation and instrument pricing notebooks.